<a href="https://colab.research.google.com/github/GarciaQuinteroAngelAlonso/Procesos-Estocasticos/blob/main/Actividad5_Uniformizacion_CMTC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Actividad 5 - Procesos Estocásticos

## Método de Uniformización para Cadenas de Markov en Tiempo Continuo

### Objetivo

En esta actividad se calcula la matriz de transición \(P(t)\) asociada a una cadena de Markov en tiempo continuo utilizando el método de uniformización.

Se realizarán dos aproximaciones:

1. Utilizando un número fijo de términos

$$
M \approx \max\{rt+5\sqrt{rt},\,20\}
$$

2. Utilizando una tolerancia de error

$$
\varepsilon = 10^{-5}
$$

Además, se verificará la ecuación de Chapman-Kolmogorov

$$
P(1)=P(0.5)P(0.5).
$$

---

## Idea del Método

Sea \(R\) la matriz de tasas del proceso.

Primero se calcula la tasa de uniformización

$$
r=\max_i r_i,
$$

donde \(r_i\) es la suma de la fila \(i\) de \(R\).

Posteriormente se construye la matriz estocástica \(\hat P\) mediante

$$
\hat P_{ij}=\frac{r_{ij}}{r}, \qquad i\neq j,
$$

y

$$
\hat P_{ii}=1-\frac{r_i}{r}.
$$

Con esta matriz, la probabilidad de transición se expresa como

$$
P(t)=
\sum_{k=0}^{\infty}
e^{-rt}
\frac{(rt)^k}{k!}
\hat P^k.
$$

La serie anterior se trunca después de un número adecuado de términos para obtener una aproximación numérica de \(P(t)\).

---

## Actividades Realizadas

### Ejercicio 3

Calcular:

- \(P(0.5)\)
- \(P(1)\)
- \(P(5)\)

empleando

$$
M=\max\{rt+5\sqrt{rt},20\}.
$$

Posteriormente verificar la ecuación de Chapman-Kolmogorov

$$
P(1)=P(0.5)P(0.5).
$$

### Ejercicio 4.2

Repetir los cálculos utilizando el algoritmo de uniformización basado en una tolerancia

$$
\varepsilon=10^{-5}.
$$

Para cada valor de \(t\):

- Determinar el valor de \(M\) requerido.
- Calcular \(P(t)\).
- Comparar los resultados con los obtenidos en el Ejercicio 3.

In [2]:

"""Angel Alonso García Quintero"""

import numpy as np
from math import sqrt, exp, ceil

# MATRIZ DE TASAS
#
# R[i,j] representa la tasa de transición del estado i al estado j.
# La suma de cada fila corresponde a la tasa total de salida.
#
# Estados: {0,1,2,3}

R = np.array([
    [0, 2, 3, 0],
    [4, 0, 2, 0],
    [0, 2, 0, 2],
    [1, 0, 3, 0]
], dtype=float)


def calcular_r_y_Phat(R):
    """
    Construye la matriz estocástica P̂ utilizada en la uniformización.

    Paso 1:
        r_i = suma de la fila i

    Paso 2:
        r = max(r_i)

    Paso 3:
        p̂_ij = r_ij / r     para i != j
        p̂_ii = 1 - r_i / r
    """
    r_vec = R.sum(axis=1)
    r = r_vec.max()

    N = R.shape[0]

    P_hat = R / r

    for i in range(N):
        P_hat[i, i] = 1 - r_vec[i] / r

    return r, P_hat


def calcular_M(r, t):
    """Calcula el número de términos de la aproximación truncada.
    M = max{rt + 5*sqrt(rt), 20}"""
    rt = r * t
    return int(ceil(max(rt + 5 * sqrt(rt), 20)))


def P_t_M_fijo(P_hat, r, t, M):
    """Aproxima P(t) utilizando M términos de la serie de uniformización."""
    N = P_hat.shape[0]

    rt = r * t
    e_rt = exp(-rt)

    Pt = np.zeros((N, N))

    Pk = np.eye(N)
    coef = e_rt

    for k in range(M + 1):

        Pt += coef * Pk

        if k < M:
            coef = coef * rt / (k + 1)
            Pk = Pk @ P_hat

    return Pt


def P_t_epsilon(P_hat, r, t, epsilon):
    """Aproxima P(t) hasta que la cola de la distribución de Poisson
    sea menor o igual que epsilon."""
    rt = r * t
    e_rt = exp(-rt)

    A = P_hat.copy()
    B = e_rt * np.eye(P_hat.shape[0])

    c = e_rt
    s = c

    k = 1

    while s < 1 - epsilon:

        c = c * rt / k
        B = B + c * A
        A = A @ P_hat

        s += c
        k += 1

    return B, k - 1


def imprimir_matriz(nombre, M, dec=6):

    print(f"\n{nombre}")
    print("-" * len(nombre))

    for fila in M:
        print("[ " + "  ".join(f"{x:.{dec}f}" for x in fila) + " ]")

r, P_hat = calcular_r_y_Phat(R)

print("=" * 70)
print("EJERCICIO 3 - MÉTODO CON M FIJO")
print("=" * 70)

# P(0.5)
M_05 = calcular_M(r, 0.5)
P_05 = P_t_M_fijo(P_hat, r, 0.5, M_05)

# P(1)
M_1 = calcular_M(r, 1.0)
P_1 = P_t_M_fijo(P_hat, r, 1.0, M_1)

# P(5)
M_5 = calcular_M(r, 5.0)
P_5 = P_t_M_fijo(P_hat, r, 5.0, M_5)

imprimir_matriz("P(0.5)", P_05)
imprimir_matriz("P(1)", P_1)
imprimir_matriz("P(5)", P_5)

# VERIFICACIÓN DE CHAPMAN-KOLMOGOROV
#
# P(t+s)=P(t)P(s)
#
# En particular:
#
# P(1)=P(0.5)P(0.5)

P_05_sq = P_05 @ P_05

error_ck = np.max(np.abs(P_1 - P_05_sq))

print("\nError Chapman-Kolmogorov:", error_ck)

# EJERCICIO 4.2

epsilon = 1e-5

print("\n" + "=" * 70)
print("EJERCICIO 4.2 - MÉTODO CON TOLERANCIA")
print("=" * 70)

P_05_e, M_05_e = P_t_epsilon(P_hat, r, 0.5, epsilon)
P_1_e, M_1_e = P_t_epsilon(P_hat, r, 1.0, epsilon)
P_5_e, M_5_e = P_t_epsilon(P_hat, r, 5.0, epsilon)

print(f"\nM utilizado para t=0.5 : {M_05_e}")
print(f"M utilizado para t=1   : {M_1_e}")
print(f"M utilizado para t=5   : {M_5_e}")

print("\nDiferencias máximas respecto al método M fijo")

print("t=0.5 :", np.max(np.abs(P_05 - P_05_e)))
print("t=1   :", np.max(np.abs(P_1 - P_1_e)))
print("t=5   :", np.max(np.abs(P_5 - P_5_e)))


EJERCICIO 3 - MÉTODO CON M FIJO

P(0.5)
------
[ 0.250609  0.216965  0.386657  0.145770 ]
[ 0.253135  0.238361  0.374409  0.134095 ]
[ 0.169119  0.193615  0.420301  0.216965 ]
[ 0.158017  0.157445  0.398332  0.286206 ]

P(1)
----
[ 0.206151  0.203902  0.398710  0.191236 ]
[ 0.208284  0.205341  0.397899  0.188474 ]
[ 0.196758  0.198379  0.400959  0.203902 ]
[ 0.192046  0.193997  0.401471  0.212484 ]

P(5)
----
[ 0.200000  0.200000  0.399999  0.200000 ]
[ 0.200000  0.200000  0.399999  0.200000 ]
[ 0.200000  0.200000  0.399999  0.200000 ]
[ 0.200000  0.200000  0.399999  0.200000 ]

Error Chapman-Kolmogorov: 5.820333638384412e-07

EJERCICIO 4.2 - MÉTODO CON TOLERANCIA

M utilizado para t=0.5 : 13
M utilizado para t=1   : 19
M utilizado para t=5   : 56

Diferencias máximas respecto al método M fijo
t=0.5 : 1.3607613894017767e-06
t=1   : 1.490024779948751e-06
t=5   : 2.200128962071002e-06
